In [ ]:
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(readr); library(ggplot2)
})

dir_d <- "/project/work/2026-09-01_GSEA_test/Data/"
dir_p <- "/project/work/2026-09-01_GSEA_test/Plot/"

fate_lv  <- c("DM", "DAL", "AAM")
ct_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
fate_colors <- c(DM = "#3C5488", DAL = "#E64B35", AAM = "#00A087")

src <- read_csv(file.path(dir_d, "2026-09-06_1_fate_module_celltype_source.csv"),
                show_col_types = FALSE)

# direction 重编码为 Central / Peripheral
src <- src %>%
  mutate(
    region   = recode(direction,
                      "Macular direction"    = "Central",
                      "Peripheral direction" = "Peripheral"),
    fate     = factor(fate,     fate_lv),
    celltype = factor(celltype, ct_order)
  )

cat("原始行数:", nrow(src), "\n")
cat("region 取值:", paste(unique(src$region), collapse = ", "), "\n")


In [ ]:
# ===== 追加(2026-09-11)：Central + Peripheral 合并为一张双分面 pie 矩阵 =====
# 目的：两 region 并入同一坐标系（共享 y 轴 module 顺序 + 共享 x 轴 celltype），
#       图例统一——单一 Fate 图例 + 单一忠实 genes 大小图例（面积 ∝ 基因数，两分面通用）。
# 仅依赖已保存的 2026-09-09_5_matrix_anchor_pie_data.csv，可单独运行本段重绘。
suppressPackageStartupMessages({
  library(dplyr); library(readr); library(ggplot2); library(scatterpie); library(ggforce)
})

dir_d <- "/project/work/2026-09-01_GSEA_test/Data/"
dir_p <- "/project/work/2026-09-01_GSEA_test/Plot/"

fate_lv     <- c("DM", "DAL", "AAM")
ct_order    <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
fate_colors <- c(DM = "#66C2A5", DAL = "#FC8D62", AAM = "#8DA0CB")  # 2026-10-19 更新；旧版 DM=#3C5488, DAL=#E64B35, AAM=#00A087
region_lv   <- c("Central", "Peripheral")

d_pie <- read_csv(file.path(dir_d, "2026-09-09_5_matrix_anchor_pie_data.csv"),
                  show_col_types = FALSE)

r_max     <- 0.45
max_total <- max(d_pie$total_n_anchor_genes)          # 12：两 region 共用半径标定

# module 顺序：两 region 共用全局并集（字母序），y 位置一一对应 → 两分面完全一致
mod_lv <- sort(unique(as.character(d_pie$module)))
d_pie  <- d_pie %>%
  mutate(y      = match(as.character(module), mod_lv),
         region = factor(region, region_lv))
n_ct  <- length(ct_order)
n_mod <- length(mod_lv)

# 忠实大小图例：以真实饼半径画 1/5/12 三个参考圆（面积 ∝ 基因数，与面板内饼一一等大）
# 竖排 + 间距，保证标注清晰；仅出现在 Peripheral 分面右侧留白处（合并后全图仅一份）
leg_tot <- c(1, 5, max_total)
leg_rad <- r_max * sqrt(leg_tot / max_total)
x_leg   <- n_ct + 1.2
gap     <- 0.55
yc      <- numeric(length(leg_tot)); yc[1] <- 1.6
for (i in 2:length(leg_tot)) yc[i] <- yc[i - 1] + leg_rad[i - 1] + leg_rad[i] + gap
lab_x   <- x_leg + max(leg_rad) + 0.30
leg_df   <- data.frame(x0 = x_leg, y0 = yc, r = leg_rad, lab = leg_tot,
                       region = factor("Peripheral", region_lv))
genes_df <- data.frame(x = x_leg - max(leg_rad), y = max(yc) + max(leg_rad) + 0.7,
                       region = factor("Peripheral", region_lv))

p <- ggplot() +
  geom_scatterpie(aes(x = x, y = y, r = radius), data = d_pie,
                  cols = fate_lv, color = NA) +
  ggforce::geom_circle(aes(x0 = x0, y0 = y0, r = r), data = leg_df,
                       inherit.aes = FALSE, fill = "grey90", color = "grey40", size = 0.3) +
  geom_text(aes(x = lab_x, y = y0, label = lab), data = leg_df,
            inherit.aes = FALSE, hjust = 0, size = 2.8, color = "black") +
  geom_text(aes(x = x, y = y), data = genes_df, label = "genes", size = 3, hjust = 0) +
  scale_fill_manual(values = fate_colors, name = "Fate", breaks = fate_lv) +
  facet_wrap(~ region, nrow = 1) +
  scale_x_continuous(breaks = seq_len(n_ct), labels = ct_order) +
  scale_y_continuous(breaks = seq_len(n_mod), labels = mod_lv) +
  coord_fixed(ratio = 1, xlim = c(0.4, n_ct + 2.3),
              ylim = c(0.4, n_mod + 0.6), clip = "off") +   # 保正圆；coord 限定可视范围（不丢弃右侧图例）
  labs(title = "Anchor-gene composition (Central vs Peripheral)",
       x = "Cell type", y = "Functional module") +
  theme_bw(base_size = 9) +
  theme(panel.grid.minor = element_blank(),
        panel.grid.major = element_line(color = "grey92"),
        panel.border     = element_rect(color = "grey30", fill = NA),
        panel.spacing    = unit(0.6, "lines"),
        strip.text       = element_text(face = "bold", size = 10),
        axis.text.x      = element_text(color = "black", size = 9, angle = 40, hjust = 1),
        axis.text.y      = element_text(color = "black", size = 7.5),
        axis.title       = element_text(color = "black", size = 9),
        plot.title       = element_text(size = 10, face = "bold", hjust = 0),
        legend.position  = "right",
        legend.title     = element_text(size = 8),
        legend.text      = element_text(size = 7.5),
        legend.key.size  = unit(0.4, "cm")) +
  guides(fill = guide_legend(order = 1, override.aes = list(size = 4)))

unit_in <- 0.24
w <- 2 * (n_ct + 2.3) * unit_in + 2.4
h <- n_mod * unit_in + 1.2
print(p)
ggsave(file.path(dir_p, "2026-09-09_5_matrix_Central_Peripheral_pie.pdf"),
       p, width = w, height = h)
cat(sprintf("已保存: 2026-09-09_5_matrix_Central_Peripheral_pie.pdf  (w=%.2f h=%.2f)\n", w, h))


In [ ]:
# ===== 追加(2026-10-19)：PanelA module 顺序版 pie 矩阵 =====
# 目的：与 2026-09-02_4_PanelA_AnchorGenes_n_byDirection_pie.pdf 的 module 排序保持一致——
#       按“出现的 DM/DAL/AAM 组数(两方向合并)从多到少、同数按锚定基因总数降序”，最共享 module 置顶。
#       配色与上一 cell 一致（DM=#66C2A5, DAL=#FC8D62, AAM=#8DA0CB）。
# 仅依赖已保存 CSV，可单独运行本段重绘。
suppressPackageStartupMessages({
  library(dplyr); library(readr); library(ggplot2); library(scatterpie); library(ggforce)
})

dir_d <- "/project/work/2026-09-01_GSEA_test/Data/"
dir_p <- "/project/work/2026-09-01_GSEA_test/Plot/"

fate_lv     <- c("DM", "DAL", "AAM")
ct_order    <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
fate_colors <- c(DM = "#66C2A5", DAL = "#FC8D62", AAM = "#8DA0CB")
region_lv   <- c("Central", "Peripheral")

# ---- PanelA module 顺序（复刻 2026-09-02_4 byDirection cell 的排序逻辑）----
.conn <- read_csv(file.path(dir_d, "2026-09-02_2_fate_pathway_connection_all.csv"),
                  show_col_types = FALSE) %>% filter(pass_U1)
.mm   <- read_csv(file.path(dir_d, "2026-09-02_3_module_map_combined.csv"),
                  show_col_types = FALSE)
.uniq <- function(x) unique(unlist(strsplit(paste(x, collapse = ";"), ";")))
.prof_of <- function(.DIR) {
  .conn %>% left_join(.mm, by = c("direction", "pathway")) %>%
    filter(direction == .DIR) %>% mutate(subclass = factor(subclass, fate_lv)) %>%
    group_by(subclass, module) %>%
    summarise(n_gene = length(.uniq(anchor_genes)), .groups = "drop")
}
.d <- bind_rows(
  .prof_of("Macular direction")    %>% mutate(tissue = "central"),
  .prof_of("Peripheral direction") %>% mutate(tissue = "peripheral")
) %>% filter(!is.na(module), n_gene > 0)
.ord <- .d %>%
  group_by(module) %>%
  summarise(n_fate = n_distinct(subclass), tot = sum(n_gene), .groups = "drop") %>%
  arrange(desc(n_fate), desc(tot))

d_pie <- read_csv(file.path(dir_d, "2026-09-09_5_matrix_anchor_pie_data.csv"),
                  show_col_types = FALSE)

r_max     <- 0.45
max_total <- max(d_pie$total_n_anchor_genes)

# y 轴自下而上与 PanelA 一致：rev(.ord) 使最共享 module 置顶；
# 兜底：矩阵数据中出现而 PanelA 未覆盖的 module 按字母序排在最下方。
mod_lv <- rev(.ord$module)
mod_lv <- c(setdiff(sort(unique(as.character(d_pie$module))), mod_lv), mod_lv)
d_pie  <- d_pie %>%
  mutate(y      = match(as.character(module), mod_lv),
         region = factor(region, region_lv))
n_ct  <- length(ct_order)
n_mod <- length(mod_lv)
cat("PanelA顺序版 y轴自下而上: ", paste(head(mod_lv, 3), collapse=" | "), "...", paste(tail(mod_lv, 3), collapse=" | "), "\n")

# 忠实大小图例：以真实饼半径画 1/5/12 三个参考圆（面积 ∝ 基因数，与面板内饼一一等大）
leg_tot <- c(1, 5, max_total)
leg_rad <- r_max * sqrt(leg_tot / max_total)
x_leg   <- n_ct + 1.2
gap     <- 0.55
yc      <- numeric(length(leg_tot)); yc[1] <- 1.6
for (i in 2:length(leg_tot)) yc[i] <- yc[i - 1] + leg_rad[i - 1] + leg_rad[i] + gap
lab_x   <- x_leg + max(leg_rad) + 0.30
leg_df   <- data.frame(x0 = x_leg, y0 = yc, r = leg_rad, lab = leg_tot,
                       region = factor("Peripheral", region_lv))
genes_df <- data.frame(x = x_leg - max(leg_rad), y = max(yc) + max(leg_rad) + 0.7,
                       region = factor("Peripheral", region_lv))

p <- ggplot() +
  geom_scatterpie(aes(x = x, y = y, r = radius), data = d_pie,
                  cols = fate_lv, color = NA) +
  ggforce::geom_circle(aes(x0 = x0, y0 = y0, r = r), data = leg_df,
                       inherit.aes = FALSE, fill = "grey90", color = "grey40", size = 0.3) +
  geom_text(aes(x = lab_x, y = y0, label = lab), data = leg_df,
            inherit.aes = FALSE, hjust = 0, size = 2.8, color = "black") +
  geom_text(aes(x = x, y = y), data = genes_df, label = "genes", size = 3, hjust = 0) +
  scale_fill_manual(values = fate_colors, name = "Fate", breaks = fate_lv) +
  facet_wrap(~ region, nrow = 1) +
  scale_x_continuous(breaks = seq_len(n_ct), labels = ct_order) +
  scale_y_continuous(breaks = seq_len(n_mod), labels = mod_lv) +
  coord_fixed(ratio = 1, xlim = c(0.4, n_ct + 2.3),
              ylim = c(0.4, n_mod + 0.6), clip = "off") +   # 保正圆；coord 限定可视范围（不丢弃右侧图例）
  labs(title = "Anchor-gene composition (Central vs Peripheral, PanelA module order)",
       x = "Cell type", y = "Functional module") +
  theme_bw(base_size = 9) +
  theme(panel.grid.minor = element_blank(),
        panel.grid.major = element_line(color = "grey92"),
        panel.border     = element_rect(color = "grey30", fill = NA),
        panel.spacing    = unit(0.6, "lines"),
        strip.text       = element_text(face = "bold", size = 10),
        axis.text.x      = element_text(color = "black", size = 9, angle = 40, hjust = 1),
        axis.text.y      = element_text(color = "black", size = 7.5),
        axis.title       = element_text(color = "black", size = 9),
        plot.title       = element_text(size = 10, face = "bold", hjust = 0),
        legend.position  = "right",
        legend.title     = element_text(size = 8),
        legend.text      = element_text(size = 7.5),
        legend.key.size  = unit(0.4, "cm")) +
  guides(fill = guide_legend(order = 1, override.aes = list(size = 4)))

unit_in <- 0.24
w <- 2 * (n_ct + 2.3) * unit_in + 2.4
h <- n_mod * unit_in + 1.2
print(p)
ggsave(file.path(dir_p, "2026-09-09_5_matrix_Central_Peripheral_pie_panelAorder.pdf"),
       p, width = w, height = h)
cat(sprintf("已保存: 2026-09-09_5_matrix_Central_Peripheral_pie_panelAorder.pdf  (w=%.2f h=%.2f)\n", w, h))
